# Profit_G_q 因子

Profit_G_q 因子：当季净利润（最新财报）同比增长率

## 裸因子指标计算

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant / DAI 复现华泰成长类因子 Profit_G_q：裸因子指标测试版 v1。

口径说明：
1. 研报 Profit_G_q = 当季净利润（最新财报）同比增长率。
2. BigQuant DAI 数据页中，财务因子不一定以 fs_ 前缀出现在 SQL 表中；
   对应字段更可能是 net_profit_mrq_yoy / net_profit_to_parent_mrq_yoy / net_profit_yoy。
3. 本代码只使用 dai.query，不调用旧版 D.features / D.financial_statements。
4. 优先使用单季度口径：net_profit_mrq_yoy，其次使用 net_profit_to_parent_mrq_yoy；
   若账号权限或字段不可用，则回退到最新一期口径 net_profit_yoy / net_profit_to_parent_yoy。
5. 裸因子版本：不做市值中性化、不做行业中性化；只做截面 MAD 去极值与标准化。
6. 每隔 rebalance_freq 个交易日取一个截面，计算未来 forward_days 个交易日收益的 IC、RankIC、WLS 因子收益率和 t 值。
7. 股票池剔除 ST、当前停牌、下一交易日停牌、北交所；收益使用 cn_stock_factors_base 的 close。
8. 回归法：未来 forward_days 日相对沪深300超额收益 ~ const + Profit_G_q 裸标准化因子，
   WLS 权重为 sqrt(流通市值)。

如果仍报字段不存在，请先运行：
    test_factor_sources(CFG)
它会逐个测试候选表和字段，并显示当前账号实际可用的 Profit_G_q 字段来源。
"""

import os
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

try:
    import dai
except ImportError:
    from bigquant import dai  # type: ignore

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class FactorSource:
    table: str
    field: str
    desc: str


@dataclass(frozen=True)
class Config:
    start_date: str = "2020-01-01"
    end_date: str = "2026-06-30"

    forward_days: int = 30
    rebalance_freq: int = 30
    min_cross_section_size: int = 30

    factor_name: str = "Profit_G_q_raw"
    chinese_font_path: str = ""

    # 这里不要使用旧 D.features 口径的 fs_net_profit_yoy_0；当前代码只走 DAI SQL。
    # 优先尝试单季度净利润同比字段；若当前账号无对应字段，则回退到净利润同比 / 归母净利润同比。
    factor_sources: Tuple[FactorSource, ...] = field(default_factory=lambda: (
        # 优先尝试“单季度净利润同比”字段；不同账号/数据表字段覆盖可能不完全一致，因此保留多组候选。
        FactorSource("cn_stock_prefactors", "net_profit_mrq_yoy", "净利润（单季度，同比增长）"),
        FactorSource("cn_stock_prefactors", "net_profit_to_parent_mrq_yoy", "归母净利润（单季度，同比增长）"),
        FactorSource("cn_stock_prefactors", "net_profit_to_parent_deducted_mrq_yoy", "扣非归母净利润（单季度，同比增长）"),
        FactorSource("cn_stock_prefactors", "net_profit_yoy", "净利润同比增长率"),
        FactorSource("cn_stock_prefactors", "net_profit_to_parent_yoy", "归属于母公司所有者的净利润同比增长率"),
        FactorSource("cn_stock_factors_financial_indicators", "net_profit_mrq_yoy", "净利润（单季度，同比增长）"),
        FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_mrq_yoy", "归母净利润（单季度，同比增长）"),
        FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_deducted_mrq_yoy", "扣非归母净利润（单季度，同比增长）"),
        FactorSource("cn_stock_factors_financial_indicators", "net_profit_yoy", "净利润同比增长率"),
        FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy", "归属于母公司所有者的净利润同比增长率"),
    ))


CFG = Config()


def test_factor_sources(cfg: Config = CFG) -> pd.DataFrame:
    """逐个测试候选 Profit_G_q 字段是否可在当前 BigQuant / DAI 环境中读取。"""
    rows: List[Dict[str, object]] = []
    for src in cfg.factor_sources:
        sql = f"""
        SELECT date, instrument, {src.field} AS factor_raw
        FROM {src.table}
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{cfg.end_date}'
          AND {src.field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = dai.query(sql, filters={"date": [cfg.start_date, cfg.end_date]}).df()
            ok = not tmp.empty
            rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": ok,
                "rows": len(tmp),
                "error": "" if ok else "查询成功但无非空样本",
            })
        except Exception as e:
            rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })
    out = pd.DataFrame(rows)
    display(out)
    return out


def _font_has_chinese(font_path: str) -> bool:
    try:
        ft = font_manager.get_font(font_path)
        cmap = ft.get_charmap()
        return all(ord(ch) in cmap for ch in "因子日期相关系数")
    except Exception:
        return False


def _candidate_font_paths(user_font_path: str = "") -> List[str]:
    candidates: List[str] = []
    if user_font_path:
        candidates.append(user_font_path)
    env_font = os.environ.get("CHINESE_FONT_PATH", "")
    if env_font:
        candidates.append(env_font)
    candidates.extend([
        "./SimHei.ttf", "./simhei.ttf", "./msyh.ttc", "./Microsoft YaHei.ttf",
        "./NotoSansCJK-Regular.ttc", "/home/jovyan/work/SimHei.ttf",
        "/home/jovyan/work/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
        "/System/Library/Fonts/PingFang.ttc",
        "C:/Windows/Fonts/msyh.ttc", "C:/Windows/Fonts/simhei.ttf",
    ])

    search_roots = [Path.cwd(), Path.home(), Path("/usr/share/fonts"), Path("/usr/local/share/fonts")]
    name_keywords = (
        "NotoSansCJK", "NotoSansSC", "SourceHanSans", "WenQuanYi", "wqy",
        "SimHei", "simhei", "msyh", "PingFang", "Arial Unicode",
    )
    suffixes = {".ttf", ".ttc", ".otf"}
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*"):
                if p.suffix.lower() in suffixes and any(k.lower() in p.name.lower() for k in name_keywords):
                    candidates.append(str(p))
        except Exception:
            continue

    seen = set()
    unique = []
    for p in candidates:
        pp = str(Path(p).expanduser())
        if pp not in seen:
            seen.add(pp)
            unique.append(pp)
    return unique


def set_chinese_font(font_path: str = "") -> Optional[font_manager.FontProperties]:
    preferred_names = [
        "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Noto Sans SC",
        "Source Han Sans SC", "WenQuanYi Micro Hei", "PingFang SC", "Arial Unicode MS",
    ]
    plt.rcParams["axes.unicode_minus"] = False
    for path in _candidate_font_paths(font_path):
        if not Path(path).exists() or not _font_has_chinese(path):
            continue
        try:
            font_manager.fontManager.addfont(path)
            prop = font_manager.FontProperties(fname=path)
            font_name = prop.get_name()
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name] + preferred_names + ["DejaVu Sans"]
            return prop
        except Exception:
            continue

    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in preferred_names:
        if name in available:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [name] + ["DejaVu Sans"]
            return font_manager.FontProperties(family=name)

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = preferred_names + ["DejaVu Sans"]
    return None


def _query_panel_with_source(cfg: Config, src: FactorSource, fetch_end: str) -> pd.DataFrame:
    """使用指定候选表和字段读取完整测试面板。"""
    sql = f"""
    PRAGMA enable_pushdown_window;

    WITH trading_dates AS (
        SELECT
            date,
            ROW_NUMBER() OVER (ORDER BY date) AS rn
        FROM (
            SELECT DISTINCT date
            FROM cn_stock_factors_base
            WHERE date >= DATE '{cfg.start_date}'
              AND date <= DATE '{cfg.end_date}'
        )
    ),

    signal_dates AS (
        SELECT date
        FROM trading_dates
        WHERE MOD(rn - 1, {cfg.rebalance_freq}) = 0
    ),

    base AS (
        SELECT
            date,
            instrument,
            close,
            float_market_cap,
            st_status,
            suspended,
            list_sector,
            LEAD(close, {cfg.forward_days}) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS close_fwd,
            LEAD(suspended, 1) OVER (
                PARTITION BY instrument ORDER BY date
            ) AS next_suspended
        FROM cn_stock_factors_base
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
          AND COALESCE(list_sector, 0) != 4
    ),

    bench_raw AS (
        SELECT
            date,
            MAX(hs300_close) AS hs300_close
        FROM cn_stock_factors_base
        WHERE date >= DATE '{cfg.start_date}'
          AND date <= DATE '{fetch_end}'
        GROUP BY date
    ),

    bench AS (
        SELECT
            date,
            hs300_close,
            LEAD(hs300_close, {cfg.forward_days}) OVER (ORDER BY date) AS hs300_close_fwd
        FROM bench_raw
    )

    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        f.{src.field} AS factor_raw,
        b.close_fwd / b.close - 1.0 AS ret_fwd,
        b.close_fwd / b.close - 1.0
            - (be.hs300_close_fwd / be.hs300_close - 1.0) AS excess_ret_fwd
    FROM base AS b
    JOIN signal_dates AS sd
      ON b.date = sd.date
    JOIN {src.table} AS f
      ON b.date = f.date AND b.instrument = f.instrument
    JOIN bench AS be
      ON b.date = be.date
    WHERE b.date <= DATE '{cfg.end_date}'
      AND b.st_status = 0
      AND b.suspended = 0
      AND COALESCE(b.next_suspended, 1) = 0
      AND b.close > 0
      AND b.close_fwd > 0
      AND be.hs300_close > 0
      AND be.hs300_close_fwd > 0
      AND b.float_market_cap > 0
      AND f.{src.field} IS NOT NULL
    ORDER BY b.date, b.instrument
    """
    return dai.query(sql, filters={"date": [cfg.start_date, fetch_end]}).df()


def fetch_signal_panel(cfg: Config) -> pd.DataFrame:
    """按候选字段顺序读取调仓截面数据；前一个字段不可用时自动尝试下一个。"""
    fetch_end = (
        pd.Timestamp(cfg.end_date) + pd.Timedelta(days=max(90, cfg.forward_days * 12))
    ).strftime("%Y-%m-%d")

    errors: List[str] = []
    chosen_source: Optional[FactorSource] = None
    df: Optional[pd.DataFrame] = None

    for src in cfg.factor_sources:
        try:
            tmp = _query_panel_with_source(cfg, src, fetch_end)
            if tmp.empty:
                errors.append(f"{src.table}.{src.field}: 查询成功但结果为空")
                continue
            chosen_source = src
            df = tmp
            break
        except Exception as e:
            short_err = str(e).split("\n")[-1]
            errors.append(f"{src.table}.{src.field}: {short_err}")
            continue

    if df is None or chosen_source is None:
        msg = [
            "Profit_G_q 因子数据获取失败。已依次尝试以下 DAI 字段：",
            *errors,
            "",
            "建议先运行 test_factor_sources(CFG)，查看当前账号实际可用字段。",
        ]
        raise RuntimeError("\n".join(msg))

    print(f"Profit_G_q 因子数据来源：{chosen_source.table}.{chosen_source.field}（{chosen_source.desc}）")

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype("category")
    for col in ["float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"]:
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")
    df = df.dropna(subset=["float_market_cap", "factor_raw", "ret_fwd", "excess_ret_fwd"])
    if df.empty:
        raise ValueError("清洗后数据为空，请检查日期区间、字段权限或股票池过滤条件。")
    return df.reset_index(drop=True)


def robust_zscore_np(x: np.ndarray) -> np.ndarray:
    """截面 MAD 去极值 + 标准化。按研报常见写法使用 median ± 5 * MAD。"""
    x = x.astype(np.float64, copy=False)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out
    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        lo, hi = med - 5.0 * mad, med + 5.0 * mad
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])
    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def add_raw_factor_z(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    factor_z = np.full(len(df), np.nan, dtype=np.float32)
    raw = df["factor_raw"].to_numpy(dtype=np.float64, copy=False)
    for _, idx in df.groupby("date", sort=False, observed=True).indices.items():
        idx_arr = np.asarray(idx)
        factor_z[idx_arr] = robust_zscore_np(raw[idx_arr]).astype(np.float32)
    df["factor_z"] = factor_z
    return df.dropna(subset=["factor_z"]).reset_index(drop=True)


def corr_np(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan
    xv = x[valid].astype(np.float64, copy=False)
    yv = y[valid].astype(np.float64, copy=False)
    xv = xv - xv.mean()
    yv = yv - yv.mean()
    denom = np.sqrt(np.dot(xv, xv) * np.dot(yv, yv))
    if not np.isfinite(denom) or denom <= 1e-18:
        return np.nan
    return float(np.dot(xv, yv) / denom)


def rank_np(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy(dtype=np.float64, copy=False)


def wls_factor_return(g: pd.DataFrame) -> Tuple[float, float]:
    y = g["excess_ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    f = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
    w = np.sqrt(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), f])
    valid = np.isfinite(y) & np.isfinite(X).all(axis=1) & np.isfinite(w) & (w > 0)
    if valid.sum() < max(30, X.shape[1] + 5):
        return np.nan, np.nan
    Xv = X[valid]
    yv = y[valid]
    wv = w[valid]
    xtwx = Xv.T @ (wv[:, None] * Xv)
    xtwy = Xv.T @ (wv * yv)
    xtwx_inv = np.linalg.pinv(xtwx, rcond=1e-12)
    beta = xtwx_inv @ xtwy
    resid = yv - Xv @ beta
    rank = np.linalg.matrix_rank(xtwx)
    dof = max(len(yv) - rank, 1)
    sigma2 = float(np.sum(wv * resid * resid) / dof)
    se = np.sqrt(np.maximum(np.diag(sigma2 * xtwx_inv), 0.0))
    factor_ret = float(beta[1])
    t_value = float(beta[1] / se[1]) if se[1] > 1e-18 else np.nan
    return factor_ret, t_value


def calc_cross_section_metrics(g: pd.DataFrame, min_n: int) -> Optional[Dict[str, float]]:
    if len(g) < min_n:
        return None
    factor = g["factor_z"].to_numpy(dtype=np.float64, copy=False)
    ret = g["ret_fwd"].to_numpy(dtype=np.float64, copy=False)
    valid = np.isfinite(factor) & np.isfinite(ret)
    if valid.sum() < min_n:
        return None
    ic = corr_np(factor[valid], ret[valid])
    rank_ic = corr_np(rank_np(factor[valid]), rank_np(ret[valid]))
    factor_ret, t_value = wls_factor_return(g)
    return {
        "date": g["date"].iloc[0],
        "样本数": int(valid.sum()),
        "IC": ic,
        "RankIC": rank_ic,
        "因子收益率": factor_ret,
        "t值": t_value,
    }


def calc_all_metrics(data: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    for _, g in data.groupby("date", sort=True, observed=True):
        row = calc_cross_section_metrics(g, cfg.min_cross_section_size)
        if row is not None:
            rows.append(row)
    if not rows:
        raise ValueError("没有足够截面可计算指标，请检查区间、股票池或 min_cross_section_size。")
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def safe_ir(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    std = s.std(ddof=1)
    if len(s) < 2 or not np.isfinite(std) or std <= 1e-18:
        return np.nan
    return float(s.mean() / std)


def make_summary(metrics: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    ic = pd.to_numeric(metrics["IC"], errors="coerce")
    rank_ic = pd.to_numeric(metrics["RankIC"], errors="coerce")
    factor_ret = pd.to_numeric(metrics["因子收益率"], errors="coerce")
    t_value = pd.to_numeric(metrics["t值"], errors="coerce")
    return pd.DataFrame([{
        "因子": cfg.factor_name,
        "起始日": metrics["date"].min().strftime("%Y-%m-%d"),
        "结束日": metrics["date"].max().strftime("%Y-%m-%d"),
        "截面数": int(metrics["date"].nunique()),
        "平均截面样本数": metrics["样本数"].mean(),
        "IC均值": ic.mean(),
        "IC标准差": ic.std(ddof=1),
        "ICIR": safe_ir(ic),
        "IC胜率": (ic > 0).mean(),
        "RankIC均值": rank_ic.mean(),
        "RankIC标准差": rank_ic.std(ddof=1),
        "RankICIR": safe_ir(rank_ic),
        "RankIC胜率": (rank_ic > 0).mean(),
        "因子收益率均值": factor_ret.mean(),
        "因子收益率标准差": factor_ret.std(ddof=1),
        "t值均值": t_value.mean(),
        "|t|均值": t_value.abs().mean(),
        "|t|>2占比": (t_value.abs() > 2).mean(),
        "t均值/t标准差": safe_ir(t_value),
    }])


def format_summary(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()
    for col in ["截面数"]:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{int(x)}")
    decimal_cols = [
        "平均截面样本数", "IC均值", "IC标准差", "ICIR", "RankIC均值", "RankIC标准差",
        "RankICIR", "因子收益率均值", "因子收益率标准差", "t值均值", "|t|均值", "t均值/t标准差",
    ]
    pct_cols = ["IC胜率", "RankIC胜率", "|t|>2占比"]
    for col in decimal_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6f}")
    for col in pct_cols:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
    return out


def plot_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"], label="IC", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"], label="RankIC", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    title = f"{cfg.factor_name}：IC 与 RankIC 时序图"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("相关系数")
        ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def plot_cumsum_ic_rankic(metrics: pd.DataFrame, cfg: Config) -> None:
    font_prop = set_chinese_font(cfg.chinese_font_path)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(metrics["date"], metrics["IC"].fillna(0).cumsum(), label="IC累计值", linewidth=1.6)
    ax.plot(metrics["date"], metrics["RankIC"].fillna(0).cumsum(), label="RankIC累计值", linewidth=1.6)
    ax.axhline(0, linewidth=1.0, linestyle="--")
    title = f"{cfg.factor_name}：IC 与 RankIC 累计曲线"
    if font_prop is not None:
        ax.set_title(title, fontproperties=font_prop)
        ax.set_xlabel("日期", fontproperties=font_prop)
        ax.set_ylabel("累计相关系数", fontproperties=font_prop)
        ax.legend(prop=font_prop)
    else:
        ax.set_title(title)
        ax.set_xlabel("日期")
        ax.set_ylabel("累计相关系数")
        ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


def main() -> Tuple[pd.DataFrame, pd.DataFrame]:
    data = fetch_signal_panel(CFG)
    data = add_raw_factor_z(data)
    metrics = calc_all_metrics(data, CFG)
    summary = make_summary(metrics, CFG)
    display(format_summary(summary))
    plot_ic_rankic(metrics, CFG)
    plot_cumsum_ic_rankic(metrics, CFG)
    return summary, metrics


summary, metrics = main()


从指标上来看该因子的效用应与Sales_G_q因子类似

## 市值分层回测

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant / DAI 策略回测：Profit_G_q 市值分组选股策略

因子口径：
1. 研报 Profit_G_q = 当季净利润（最新财报）同比增长率。
2. 当前 BigQuant / DAI 环境中，不再调用旧版 D.features / D.financial_statements，
   只使用 dai.query 从 DAI SQL 表读取数据。
3. 因子字段优先级：
   - net_profit_yoy_mrq / net_profit_mrq_yoy：净利润同比增长率（单季度），更贴近 Profit_G_q；
   - net_profit_yoy_lf / net_profit_lf_yoy：净利润同比增长率（最新一期），作为回退口径。
4. 默认使用裸 Profit_G_q：每个信号截面对原始因子做 MAD 去极值 + 标准化。
5. 若 FACTOR_MODE = "neutral"，则先对原始因子做 MAD 去极值 + 标准化，
   再对 log(流通市值) 与行业哑变量做截面中性化，最后对残差再次标准化。

策略逻辑：
1. 每 REBALANCE_DAYS 个交易日生成一次信号，信号使用 signal_date 当日已经可得的数据。
2. 全市场按流通市值从小到大划分 N_SIZE_GROUPS 组，1=最小市值组，N_SIZE_GROUPS=最大市值组。
3. 通过 SIZE_GROUPS_TO_TRADE 指定参与交易的市值组，例如 [1, 2, 3, 4]。
4. 在每个指定市值组内，选取 Profit_G_q 因子最高的前 TOP_PCT 股票。
5. 所有入选股票等权配置。
6. 信号日后一交易日执行调仓；不在选股阶段读取执行日行情，避免未来函数。
7. 执行日使用当日开盘涨跌停状态做交易约束：开盘涨停不买入，开盘跌停不卖出；停牌不交易。
8. 股票池剔除 ST、*ST、当前停牌、北交所；考虑交易成本；使用 BigTrader 原生回测引擎。

使用建议：
- 成长类因子偏中低频，默认 REBALANCE_DAYS = 30。若要对比短周期，可改为 10 / 20。
- 默认 FACTOR_MODE = "raw"，用于观察 Profit_G_q 在不同市值组中的原始成长效应；
  若要测试市值行业中性化后版本，将 FACTOR_MODE 改为 "neutral"。
- 若字段读取失败，先运行 test_factor_sources() 检查当前账号实际可用字段。
"""

import gc
import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import dai
except Exception as e:
    raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 dai。") from e

try:
    from bigquant import bigtrader
except Exception:
    try:
        import bigtrader
    except Exception as e:
        raise ImportError("请在 BigQuant Notebook 环境中运行，本代码依赖 BigTrader。") from e

warnings.filterwarnings("ignore")


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

# raw：裸 Profit_G_q；neutral：市值行业中性化后的 Profit_G_q
FACTOR_MODE = "raw"  # 可改为 "neutral"
RAW_FACTOR_NAME = "Profit_G_q_raw"
NEUTRAL_FACTOR_NAME = "Profit_G_q_neutral"
FACTOR_SCORE_COL = RAW_FACTOR_NAME if FACTOR_MODE == "raw" else NEUTRAL_FACTOR_NAME

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [1, 2, 3]
N_SIZE_GROUPS = 15
TOP_PCT = 0.15
REBALANCE_DAYS = 30
MIN_STOCKS_PER_SIZE_GROUP = 20
SELECT_COUNT_METHOD = "floor"  # floor 或 ceil；至少选 1 只

# 中性化与标准化
WINSOR_MAD_N = 3.0
MIN_CROSS_SECTION_SIZE = 100
INDUSTRY_COL = "sw2021_level1"  # 可改 sw2014_level1 / cs_level1，取决于账号字段权限

# 回测参数
CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 执行日涨跌停判断容忍误差
LIMIT_EPS = 1e-4

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

if FACTOR_MODE not in {"raw", "neutral"}:
    raise ValueError("FACTOR_MODE 只能是 'raw' 或 'neutral'。")
if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")
SIZE_GROUPS_TO_TRADE = sorted(set(int(x) for x in SIZE_GROUPS_TO_TRADE))
bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if g < 1 or g > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")
if SELECT_COUNT_METHOD not in {"floor", "ceil"}:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


@dataclass(frozen=True)
class FactorSource:
    table: str
    field: str
    desc: str


FACTOR_SOURCES: Tuple[FactorSource, ...] = (
    # 优先尝试“单季度 / MRQ”口径，更贴近研报 Profit_G_q：当季净利润同比增长率。
    FactorSource("cn_stock_prefactors", "net_profit_yoy_mrq", "净利润同比增长率（单季度）"),
    FactorSource("cn_stock_prefactors", "net_profit_mrq_yoy", "净利润（单季度，同比增长）"),
    FactorSource("cn_stock_prefactors", "net_profit_to_parent_yoy_mrq", "归母净利润同比增长率（单季度）"),
    FactorSource("cn_stock_prefactors", "net_profit_to_parent_mrq_yoy", "归母净利润（单季度，同比增长）"),

    # 若当前账号没有单季度字段，则回退到最新一期口径。注意这可能与严格“当季”口径略有差异。
    FactorSource("cn_stock_prefactors", "net_profit_yoy_lf", "净利润同比增长率（最新一期）"),
    FactorSource("cn_stock_prefactors", "net_profit_lf_yoy", "净利润（最新一期，同比增长）"),
    FactorSource("cn_stock_prefactors", "net_profit_to_parent_yoy_lf", "归母净利润同比增长率（最新一期）"),
    FactorSource("cn_stock_prefactors", "net_profit_to_parent_lf_yoy", "归母净利润（最新一期，同比增长）"),

    # BigQuant 官网提示财务因子字段也可参考 cn_stock_factors_financial_indicators，故同步尝试该表。
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_yoy_mrq", "净利润同比增长率（单季度）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_mrq_yoy", "净利润（单季度，同比增长）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy_mrq", "归母净利润同比增长率（单季度）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_mrq_yoy", "归母净利润（单季度，同比增长）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_yoy_lf", "净利润同比增长率（最新一期）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_lf_yoy", "净利润（最新一期，同比增长）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy_lf", "归母净利润同比增长率（最新一期）"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_lf_yoy", "归母净利润（最新一期，同比增长）"),

    # 部分环境只暴露无后缀字段；作为最后兜底。
    FactorSource("cn_stock_prefactors", "net_profit_yoy", "净利润同比增长率"),
    FactorSource("cn_stock_prefactors", "net_profit_to_parent_yoy", "归母净利润同比增长率"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_yoy", "净利润同比增长率"),
    FactorSource("cn_stock_factors_financial_indicators", "net_profit_to_parent_yoy", "归母净利润同比增长率"),
)


# =========================
# 2. 通用工具函数
# =========================

_T0 = time.time()


def _elapsed() -> str:
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg: str) -> None:
    print(f"[{_elapsed()}] {msg}", flush=True)


def query_df(sql: str, filters: Optional[dict] = None) -> pd.DataFrame:
    if filters is None:
        return dai.query(sql).df()
    return dai.query(sql, filters=filters).df()


def date_in_sql(dates) -> str:
    return ", ".join(f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates)


def downcast_float(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def robust_zscore_np(x: np.ndarray, mad_n: float = 3.0) -> np.ndarray:
    """截面 MAD 去极值后标准化。"""
    x = np.asarray(x, dtype=np.float64)
    out = np.full(x.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(x)
    if valid.sum() < 3:
        return out

    xv = x[valid]
    med = np.nanmedian(xv)
    mad = np.nanmedian(np.abs(xv - med))
    if np.isfinite(mad) and mad > 1e-12:
        scale = 1.4826 * mad
        lo, hi = med - mad_n * scale, med + mad_n * scale
    else:
        lo, hi = np.nanpercentile(xv, [1.0, 99.0])

    xv = np.clip(xv, lo, hi)
    std = xv.std(ddof=0)
    if np.isfinite(std) and std > 1e-12:
        out[valid] = (xv - xv.mean()) / std
    return out


def industry_dummies(industry: pd.Series) -> np.ndarray:
    """行业哑变量，drop_first=True。"""
    codes = pd.Categorical(industry.astype(str)).codes
    n = len(codes)
    k = int(codes.max()) + 1
    if k <= 1:
        return np.empty((n, 0), dtype=np.float64)
    mat = np.zeros((n, k - 1), dtype=np.float64)
    rows = np.arange(n)
    mask = codes > 0
    mat[rows[mask], codes[mask] - 1] = 1.0
    return mat


def add_factor_score(g: pd.DataFrame) -> pd.DataFrame:
    """
    单个截面内生成策略使用的因子列：
    - FACTOR_MODE='raw'：原始 Profit_G_q 做 MAD 去极值 + 标准化；
    - FACTOR_MODE='neutral'：原始 Profit_G_q 标准化后，对 log(流通市值) 与行业做中性化，再标准化残差。
    """
    g = g.copy().sort_values("instrument", kind="mergesort").reset_index(drop=True)
    if len(g) < MIN_CROSS_SECTION_SIZE:
        return pd.DataFrame()

    raw = g[RAW_FACTOR_NAME].to_numpy(dtype=np.float64, copy=False)
    raw_z = robust_zscore_np(raw, WINSOR_MAD_N)
    g[RAW_FACTOR_NAME] = raw_z.astype(np.float32)

    if FACTOR_MODE == "raw":
        out_cols = ["date", "instrument", "float_market_cap", RAW_FACTOR_NAME]
        out = g[out_cols].dropna(subset=[RAW_FACTOR_NAME, "float_market_cap"])
        return out

    log_cap = np.log(g["float_market_cap"].to_numpy(dtype=np.float64, copy=False).clip(min=1.0))
    dummies = industry_dummies(g["industry_level1"].fillna("未知"))
    X = np.column_stack([np.ones(len(g), dtype=np.float64), log_cap, dummies])

    valid = np.isfinite(raw_z) & np.isfinite(X).all(axis=1)
    if valid.sum() < max(MIN_CROSS_SECTION_SIZE, X.shape[1] + 5):
        return pd.DataFrame()

    resid = np.full(len(g), np.nan, dtype=np.float64)
    beta = np.linalg.lstsq(X[valid], raw_z[valid], rcond=None)[0]
    resid[valid] = raw_z[valid] - X[valid] @ beta
    neutral_z = robust_zscore_np(resid, WINSOR_MAD_N)

    g[NEUTRAL_FACTOR_NAME] = neutral_z.astype(np.float32)
    out = g[["date", "instrument", "float_market_cap", NEUTRAL_FACTOR_NAME]].copy()
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "float_market_cap"])
    return out


def calc_select_count(n: int, pct: float) -> int:
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def get_current_date_from_engine(context, data) -> Optional[str]:
    if data is not None and hasattr(data, "current_dt"):
        try:
            return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
        except Exception:
            pass
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            try:
                v = getattr(context, attr)
                if v is not None:
                    return pd.to_datetime(v).strftime("%Y-%m-%d")
            except Exception:
                pass
    return None


def get_positions_dict(context) -> Dict:
    for method in ["get_positions", "get_account_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    try:
        return context.portfolio.positions
    except Exception:
        return {}


def position_amount(pos_obj) -> float:
    for attr in ["amount", "quantity", "volume", "position"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("amount", 0))
    except Exception:
        return 0.0


def position_market_value(pos_obj) -> float:
    for attr in ["market_value", "value"]:
        try:
            return float(getattr(pos_obj, attr))
        except Exception:
            pass
    try:
        return float(pos_obj.get("market_value", 0))
    except Exception:
        return 0.0


def portfolio_value(context) -> float:
    for attr in ["portfolio_value", "total_value", "market_value"]:
        try:
            v = float(getattr(context.portfolio, attr))
            if np.isfinite(v) and v > 0:
                return v
        except Exception:
            pass
    return np.nan


def order_to_target_percent(context, instrument: str, weight: float) -> bool:
    for method in ["order_target_percent", "order_percent"]:
        if hasattr(context, method):
            try:
                getattr(context, method)(instrument, float(weight))
                return True
            except Exception:
                continue
    print(f"下单失败：找不到可用的目标仓位下单函数，{instrument}, target={weight:.6f}", flush=True)
    return False


def test_factor_sources() -> pd.DataFrame:
    """逐个测试候选 Profit_G_q 字段是否可在当前 BigQuant / DAI 环境中读取。"""
    rows: List[Dict[str, object]] = []
    for src in FACTOR_SOURCES:
        sql = f"""
        SELECT date, instrument, {src.field} AS factor_raw
        FROM {src.table}
        WHERE date >= DATE '{START_DATE}'
          AND date <= DATE '{END_DATE}'
          AND {src.field} IS NOT NULL
        LIMIT 5
        """
        try:
            tmp = query_df(sql, filters={"date": [START_DATE, END_DATE]})
            ok = not tmp.empty
            rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": ok,
                "rows": len(tmp),
                "error": "" if ok else "查询成功但无非空样本",
            })
        except Exception as e:
            rows.append({
                "table": src.table,
                "field": src.field,
                "desc": src.desc,
                "available": False,
                "rows": 0,
                "error": str(e).split("\n")[-1][:240],
            })
    out = pd.DataFrame(rows)
    display(out)
    return out


# =========================
# 3. 交易日、信号日、执行日
# =========================

progress("开始获取交易日与调仓日期")
trade_dates_sql = f"""
SELECT DISTINCT date
FROM cn_stock_factors_base
WHERE date >= DATE '{START_DATE}'
  AND date <= DATE '{END_DATE}'
ORDER BY date
"""
trade_dates_df = query_df(trade_dates_sql, filters={"date": [START_DATE, END_DATE]})
trade_dates = pd.to_datetime(trade_dates_df["date"]).drop_duplicates().sort_values().reset_index(drop=True)
if len(trade_dates) < REBALANCE_DAYS + 2:
    raise ValueError("指定时间段内交易日过少，无法完成回测。")

signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()
signal_to_execution = {}
for dt in signal_dates:
    idx_arr = trade_dates[trade_dates == dt].index
    if len(idx_arr) == 0:
        continue
    next_idx = int(idx_arr[0]) + 1
    if next_idx < len(trade_dates):
        signal_to_execution[pd.to_datetime(dt).strftime("%Y-%m-%d")] = pd.to_datetime(trade_dates.iloc[next_idx]).strftime("%Y-%m-%d")

if not signal_to_execution:
    raise ValueError("没有可用的信号日/执行日映射。")

signal_dates = [pd.to_datetime(x) for x in signal_to_execution.keys()]
execution_dates = [pd.to_datetime(x) for x in signal_to_execution.values()]
signal_date_sql = date_in_sql(signal_dates)
execution_date_sql = date_in_sql(execution_dates)

progress(f"交易日数量：{len(trade_dates):,}；信号截面数量：{len(signal_dates):,}；调仓周期：{REBALANCE_DAYS} 个交易日")
progress(f"因子版本：{FACTOR_MODE}；参与交易市值组：{SIZE_GROUPS_TO_TRADE}；每组选择因子最高前 {TOP_PCT:.2%}")


# =========================
# 4. 读取信号截面数据
# =========================


def query_signal_panel_with_source(src: FactorSource) -> pd.DataFrame:
    signal_sql = f"""
    SELECT
        b.date,
        b.instrument,
        b.float_market_cap,
        b.{INDUSTRY_COL} AS industry_level1,
        f.{src.field} AS {RAW_FACTOR_NAME}
    FROM cn_stock_factors_base AS b
    JOIN {src.table} AS f
      ON b.date = f.date AND b.instrument = f.instrument
    WHERE b.date IN ({signal_date_sql})
      AND COALESCE(b.list_sector, 0) != 4
      AND b.st_status = 0
      AND b.suspended = 0
      AND b.float_market_cap > 0
      AND f.{src.field} IS NOT NULL
    ORDER BY b.date, b.instrument
    """
    return query_df(signal_sql, filters={"date": [START_DATE, END_DATE]})


progress("开始读取信号截面数据")
errors: List[str] = []
chosen_source: Optional[FactorSource] = None
signal_panel: Optional[pd.DataFrame] = None

for src in FACTOR_SOURCES:
    try:
        tmp = query_signal_panel_with_source(src)
        if tmp.empty:
            errors.append(f"{src.table}.{src.field}: 查询成功但结果为空")
            continue
        chosen_source = src
        signal_panel = tmp
        break
    except Exception as e:
        errors.append(f"{src.table}.{src.field}: {str(e).split(chr(10))[-1]}")
        continue

if signal_panel is None or chosen_source is None:
    raise RuntimeError(
        "Profit_G_q 因子数据获取失败。已依次尝试以下 DAI 字段：\n"
        + "\n".join(errors)
        + "\n\n建议先运行 test_factor_sources()，查看当前账号实际可用字段。"
    )

progress(f"Profit_G_q 因子数据来源：{chosen_source.table}.{chosen_source.field}（{chosen_source.desc}）")

signal_panel["date"] = pd.to_datetime(signal_panel["date"]).dt.normalize()
signal_panel["instrument"] = signal_panel["instrument"].astype(str)
signal_panel["industry_level1"] = signal_panel["industry_level1"].fillna("未知").astype(str)
signal_panel = downcast_float(signal_panel, ["float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel.dropna(subset=["date", "instrument", "float_market_cap", RAW_FACTOR_NAME])
signal_panel = signal_panel[(signal_panel["float_market_cap"] > 0) & np.isfinite(signal_panel[RAW_FACTOR_NAME])]
signal_panel = signal_panel.drop_duplicates(subset=["date", "instrument"], keep="last")
progress(f"信号截面数据：{len(signal_panel):,} 行")


# =========================
# 5. 因子处理、市值15组、组内 Top 10% 选股
# =========================

progress("开始逐截面因子处理、市值分层与选股")
selected_parts = []
n_dates = signal_panel["date"].nunique()

for i, (dt, g) in enumerate(signal_panel.groupby("date", sort=True), 1):
    if i == 1 or i % 10 == 0 or i == n_dates:
        progress(f"处理截面 {i}/{n_dates}：{pd.to_datetime(dt).strftime('%Y-%m-%d')}，样本 {len(g):,}")

    factor_df = add_factor_score(g)
    if factor_df.empty:
        continue

    factor_df = factor_df.sort_values(["float_market_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = factor_df["float_market_cap"].rank(method="first", ascending=True)
    try:
        factor_df["size_group"] = pd.qcut(rank, q=N_SIZE_GROUPS, labels=list(range(1, N_SIZE_GROUPS + 1))).astype(int)
    except Exception:
        continue

    group_selected = []
    for sg_id in SIZE_GROUPS_TO_TRADE:
        sg = factor_df[factor_df["size_group"] == sg_id].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue
        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([FACTOR_SCORE_COL, "instrument"], ascending=[False, True], kind="mergesort")
        group_selected.append(sg.head(n_select))

    if group_selected:
        selected_parts.append(pd.concat(group_selected, ignore_index=True))

if not selected_parts:
    raise ValueError("没有形成任何有效选股结果，请检查市值组、样本数量、因子数据或 FACTOR_MODE。")

selected_df = pd.concat(selected_parts, ignore_index=True)
del selected_parts, signal_panel
gc.collect()

selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
selected_df = selected_df.dropna(subset=["execution_date"]).copy()
selected_df["stock_count"] = selected_df.groupby("signal_date")["instrument"].transform("count")
selected_df = selected_df[selected_df["stock_count"] > 0].copy()
selected_df["target_weight"] = 1.0 / selected_df["stock_count"]

signal_df = selected_df[["signal_date", "execution_date", "instrument", "size_group", FACTOR_SCORE_COL, "target_weight"]].copy()
signal_df = signal_df.sort_values(
    ["execution_date", "size_group", FACTOR_SCORE_COL, "instrument"],
    ascending=[True, True, False, True],
    kind="mergesort",
).reset_index(drop=True)

progress(f"最终信号：{len(signal_df):,} 行；涉及股票 {signal_df['instrument'].nunique():,} 只")

signal_summary = (
    signal_df.groupby(["signal_date", "execution_date"], sort=True)
    .agg(stock_count=("instrument", "count"), avg_weight=("target_weight", "mean"))
    .reset_index()
)
progress("交易信号摘要前20行：")
display(signal_summary.head(20))

size_group_summary = (
    signal_df.groupby("size_group", sort=True)
    .agg(
        selected_rows=("instrument", "count"),
        unique_stocks=("instrument", "nunique"),
        avg_factor=(FACTOR_SCORE_COL, "mean"),
    )
    .reset_index()
)
progress("各市值组入选情况：")
display(size_group_summary)


# =========================
# 6. 读取执行日交易约束：开盘涨跌停、停牌
# =========================

progress("开始读取执行日交易约束")
trade_status_sql = f"""
SELECT
    date,
    instrument,
    open,
    upper_limit,
    lower_limit,
    suspended,
    st_status
FROM cn_stock_factors_base
WHERE date IN ({execution_date_sql})
ORDER BY date, instrument
"""
trade_status = query_df(trade_status_sql, filters={"date": [START_DATE, END_DATE]})
if trade_status.empty:
    raise ValueError("执行日交易约束数据为空，请检查 cn_stock_factors_base 字段或日期。")

trade_status["date"] = pd.to_datetime(trade_status["date"]).dt.strftime("%Y-%m-%d")
trade_status["instrument"] = trade_status["instrument"].astype(str)
trade_status = downcast_float(trade_status, ["open", "upper_limit", "lower_limit"])
trade_status["suspended"] = pd.to_numeric(trade_status["suspended"], errors="coerce").fillna(1).astype(int)
trade_status["st_status"] = pd.to_numeric(trade_status["st_status"], errors="coerce").fillna(0).astype(int)
trade_status = trade_status.drop_duplicates(subset=["date", "instrument"], keep="last")

valid_price = (
    np.isfinite(trade_status["open"]) &
    np.isfinite(trade_status["upper_limit"]) &
    np.isfinite(trade_status["lower_limit"]) &
    (trade_status["open"] > 0) &
    (trade_status["upper_limit"] > 0) &
    (trade_status["lower_limit"] > 0)
)
trade_status["open_limit_up"] = valid_price & (trade_status["open"] >= trade_status["upper_limit"] * (1.0 - LIMIT_EPS))
trade_status["open_limit_down"] = valid_price & (trade_status["open"] <= trade_status["lower_limit"] * (1.0 + LIMIT_EPS))
trade_status["can_buy_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_up"])
trade_status["can_sell_open"] = (trade_status["suspended"] == 0) & (~trade_status["open_limit_down"])

trade_status_by_date = {
    d: g.set_index("instrument")[["can_buy_open", "can_sell_open", "suspended", "open_limit_up", "open_limit_down"]].to_dict("index")
    for d, g in trade_status.groupby("date", sort=False)
}

del trade_status
gc.collect()


# =========================
# 7. BigTrader 原生回测
# =========================

progress("开始准备 BigTrader 回测输入")
backtest_data = signal_df[["execution_date", "instrument", "target_weight"]].copy()
backtest_data = backtest_data.rename(columns={"execution_date": "date"})
backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
backtest_data["instrument"] = backtest_data["instrument"].astype(str)

signal_by_execution_date = {
    d: g[["instrument", "target_weight"]].copy()
    for d, g in backtest_data.groupby("date", sort=True)
}

target_by_execution_date = {
    d: set(g["instrument"].astype(str))
    for d, g in backtest_data.groupby("date", sort=True)
}

progress("开始运行 BigTrader 原生回测")


def initialize(context):
    try:
        context.set_commission(
            bigtrader.PerOrder(
                buy_cost=BUY_COST,
                sell_cost=SELL_COST,
                min_cost=MIN_COMMISSION,
            )
        )
    except Exception as e:
        print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

    context.signal_by_execution_date = signal_by_execution_date
    context.target_by_execution_date = target_by_execution_date
    context.trade_status_by_date = trade_status_by_date
    context.rebalance_dates = set(signal_by_execution_date.keys())

    try:
        context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
    except Exception:
        pass


def _get_trade_flags(context, current_date: str, instrument: str) -> Tuple[bool, bool]:
    row = context.trade_status_by_date.get(current_date, {}).get(str(instrument))
    if row is None:
        return False, False
    return bool(row.get("can_buy_open", False)), bool(row.get("can_sell_open", False))


def handle_data(context, data):
    current_date = get_current_date_from_engine(context, data)
    if current_date is None or current_date not in context.rebalance_dates:
        return

    today_signal = context.signal_by_execution_date.get(current_date)
    if today_signal is None or len(today_signal) == 0:
        return

    target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
    target_instruments = set(target_weights.keys())
    positions = get_positions_dict(context)

    holding_instruments = set()
    current_weights = {}
    pv = portfolio_value(context)
    for ins, pos in positions.items():
        ins = str(ins)
        amt = position_amount(pos)
        if amt <= 0:
            continue
        holding_instruments.add(ins)
        mv = position_market_value(pos)
        if np.isfinite(pv) and pv > 0 and np.isfinite(mv):
            current_weights[ins] = mv / pv

    # 先卖出不在目标池中的股票；开盘跌停或停牌则不卖。
    for ins in sorted(holding_instruments - target_instruments):
        _, can_sell = _get_trade_flags(context, current_date, ins)
        if can_sell:
            order_to_target_percent(context, ins, 0.0)

    # 再调整目标股票。买入方向要求非开盘涨停；卖出方向要求非开盘跌停。
    for ins in sorted(target_weights.keys()):
        target_w = float(target_weights[ins])
        can_buy, can_sell = _get_trade_flags(context, current_date, ins)
        cur_w = current_weights.get(ins, 0.0)

        if cur_w <= 1e-8:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w > cur_w + 1e-5:
            if can_buy:
                order_to_target_percent(context, ins, target_w)
        elif target_w < cur_w - 1e-5:
            if can_sell:
                order_to_target_percent(context, ins, target_w)
        else:
            pass


run_kwargs = dict(
    data=backtest_data,
    start_date=min(signal_by_execution_date.keys()),
    end_date=END_DATE,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=CAPITAL_BASE,
    benchmark=BENCHMARK,
)

try:
    run_kwargs["market"] = bigtrader.Market.CN_STOCK
except Exception:
    pass

try:
    run_kwargs["frequency"] = bigtrader.Frequency.DAILY
except Exception:
    run_kwargs["frequency"] = "1d"

performance = bigtrader.run(**run_kwargs)

progress("BigTrader 回测完成")
try:
    display(performance.summary)
except Exception:
    display(performance)


从回测结果上来看，该因子的效用会比Sales_G_q要略强一些，但本质上的差别并不大